# Notebook 01 - Veri Kesfi

Bu notebook kapsaminda Amasya ilindeki Hamamozu, Gumushacikoy ve Goynucek ilcelerine ait elektrik tuketim ve tahsilat verileri incelenmektedir.

## Icindekiler

1. Veri Setinin Yuklenmesi
2. Veri Seti Genel Ozeti
3. Temel Pandas Kontrolleri
4. Eksik Deger Kontrolu
5. Tahakkuk Verilerinin Birlestirilmesi
6. Ilce Bazinda Benzersiz Musteri Sayilari
7. kWh Tuketim Verisinin Kalite Kontrolu
8. Negatif ve Sifir Tuketim Yorumu
9. Aykiri Deger Analizi
10. Hesap Sinifina Gore Tuketim Istatistikleri
11. Analize Hazirlik
12. Genel Degerlendirme

### Amaclar
- Veri setlerinin yapisini ve temel ozelliklerini ozetlemek
- Eksik deger, veri tipi ve yinelenen kayit kontrollerini yapmak
- Ilce bazinda benzersiz musteri sayilarini karsilastirmak
- Tahakkuk verilerini tek bir veri setinde birlestirmek
- kWh tuketim verilerindeki negatif, sifir ve aykiri degerleri incelemek
- Hesap siniflarina gore tuketim istatistiklerini analiz etmek


In [30]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")


## 1. Veri Setinin Yuklenmesi

Excel dosyasinda bulunan tahsilat ve tahakkuk sayfalari ayri DataFrame'lere yuklenmistir.


In [31]:
file_path = "../data/elektrik_veri_hashed.xlsx"

xls = pd.ExcelFile(file_path)

print("Excel sayfalari:")
print(xls.sheet_names)


Excel sayfalari:
['Tahsilat', 'Tahsilat 1', 'Tahakkuk', 'Tahakkuk 1', 'Tahakkuk 2']


In [32]:
df_tahsilat = pd.read_excel(xls, sheet_name="Tahsilat")
df_tahsilat_1 = pd.read_excel(xls, sheet_name="Tahsilat 1")

df_hamamozu = pd.read_excel(xls, sheet_name="Tahakkuk")
df_gumushacikoy = pd.read_excel(xls, sheet_name="Tahakkuk 1")
df_goynucek = pd.read_excel(xls, sheet_name="Tahakkuk 2")

dataframes = {
    "Tahsilat": df_tahsilat,
    "Tahsilat 1": df_tahsilat_1,
    "Hamamozu": df_hamamozu,
    "Gumushacikoy": df_gumushacikoy,
    "Goynucek": df_goynucek,
}


## 2. Veri Seti Genel Ozeti

Tum veri setleri icin satir/sutun sayisi, eksik deger durumu ve degisken tipleri tek tabloda ozetlenmistir. Boylece notebook ciktisi gereksiz uzun `info()` ve `describe()` bloklariyla kalabaliklasmaz.


In [33]:
veri_ozeti = []

for name, df in dataframes.items():
    veri_ozeti.append({
        "Veri Seti": name,
        "Satir Sayisi": len(df),
        "Sutun Sayisi": df.shape[1],
        "Toplam Eksik Deger": df.isna().sum().sum(),
        "Sayisal Sutun": df.select_dtypes(include="number").shape[1],
        "Tarih Sutunu": df.select_dtypes(include="datetime").shape[1],
        "Metin/Kategorik Sutun": df.select_dtypes(include=["object", "string"]).shape[1],
    })

df_veri_ozeti = pd.DataFrame(veri_ozeti)
display(df_veri_ozeti)


,Veri Seti,Satir Sayisi,Sutun Sayisi,Toplam Eksik Deger,Sayisal Sutun,Tarih Sutunu,Metin/Kategorik Sutun
0,Tahsilat,636993,9,1910974,5,1,3
1,Tahsilat 1,917632,22,13478526,18,0,4
2,Hamamozu,124818,10,0,2,0,8
3,Gumushacikoy,765657,10,0,2,0,8
4,Goynucek,295223,10,0,2,0,8


In [34]:
kolon_ozeti = pd.DataFrame({
    "Veri Seti": list(dataframes.keys()),
    "Kolonlar": [", ".join(df.columns.astype(str)) for df in dataframes.values()]
})

display(kolon_ozeti)


,Veri Seti,Kolonlar
0,Tahsilat,"Şube, Kasa, İlçe, Söz.hsp.(bağımsız), Tahsilat..."
1,Tahsilat 1,"Mali yıl/dönem, İl, İlçe, Söz.hsp.(bağımsız), ..."
2,Hamamozu,"il, ilce, sozlesme_hesap_no, mali_yil_donem, f..."
3,Gumushacikoy,"il, ilce, sozlesme_hesap_no, mali_yil_donem, f..."
4,Goynucek,"il, ilce, sozlesme_hesap_no, mali_yil_donem, f..."


## 2.1. Temel Pandas Kontrolleri

Yonergede belirtilen `.head()`, `.info()` ve `.describe()` kontrolleri ornek olarak bir tahakkuk veri seti uzerinden uygulanmistir. Tum veri setleri icin satir/sutun ve eksik deger ozetleri bir onceki tabloda topluca verildigi icin burada cikti kalabaligini azaltmak amaclanmistir.


In [35]:
ornek_kontrol_df = df_hamamozu

print("Ornek veri seti: Hamamozu / Tahakkuk")
display(ornek_kontrol_df.head())


Ornek veri seti: Hamamozu / Tahakkuk


,il,ilce,sozlesme_hesap_no,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı,kwh
0,AMASYA,HAMAMÖZÜ,917576806,2023-01-01,2023-01-12,2023-03-06,2023-01-23,M001,Mesken,1.79
1,AMASYA,HAMAMÖZÜ,917576806,2023-01-01,2023-02-09,2023-05-11,2023-02-20,M001,Mesken,2.60
2,AMASYA,HAMAMÖZÜ,917576806,2023-02-01,2023-02-09,2023-05-11,2023-02-20,M001,Mesken,1.23
3,AMASYA,HAMAMÖZÜ,917576806,2023-02-01,2023-03-10,2023-05-11,2023-03-20,M001,Mesken,2.56
4,AMASYA,HAMAMÖZÜ,917576806,2023-03-01,2023-03-10,2023-05-11,2023-03-20,M001,Mesken,1.35


In [36]:
print("Hamamozu / Tahakkuk veri yapisi")
ornek_kontrol_df.info()


Hamamozu / Tahakkuk veri yapisi
<class 'pandas.DataFrame'>
RangeIndex: 124818 entries, 0 to 124817
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   il                 124818 non-null  str    
 1   ilce               124818 non-null  str    
 2   sozlesme_hesap_no  124818 non-null  int64  
 3   mali_yil_donem     124818 non-null  str    
 4   fatura_tarihi      124818 non-null  str    
 5   kayit_tarihi       124818 non-null  str    
 6   vade_tarihi        124818 non-null  str    
 7   hesap_sinifi       124818 non-null  object 
 8   Hesap Sınıfı       124818 non-null  str    
 9   kwh                124818 non-null  float64
dtypes: float64(1), int64(1), object(1), str(7)
memory usage: 9.5+ MB


In [37]:
print("Hamamozu / Tahakkuk sayisal tanimlayici istatistikler")
display(ornek_kontrol_df.describe())

print("Hamamozu / Tahakkuk kategorik tanimlayici istatistikler")
display(ornek_kontrol_df.describe(include=["object", "string"]))


Hamamozu / Tahakkuk sayisal tanimlayici istatistikler


,sozlesme_hesap_no,kwh
count,"124,818.00","124,818.00"
mean,"5,044,916,479.96",70.87
std,"2,874,543,657.76",389.22
min,"2,903,944.00","-1,242.99"
25%,"2,577,471,102.00",15.49
50%,"5,027,441,889.00",40.56
75%,"7,594,089,979.00",70.43
max,"9,991,894,452.00","25,941.60"


Hamamozu / Tahakkuk kategorik tanimlayici istatistikler


,il,ilce,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı
count,124818,124818,124818,124818,124818,124818,124818,124818
unique,1,1,31,451,30,423,28,28
top,AMASYA,HAMAMÖZÜ,2024-08-01,2024-07-12,2024-10-08,2024-07-22,M001,Mesken
freq,124818,124818,4222,4734,9437,4703,110682,110682


## 3. Eksik Deger Kontrolu

Eksik degerler veri seti bazinda incelenmistir. Tahsilat verilerindeki bazi eksik alanlar odeme turu veya odeme zaman araliginda tutar bulunmadigini temsil etmektedir.


In [38]:
def eksik_deger_ozeti(df, veri_adi):
    eksik = df.isna().sum()
    oran = (eksik / len(df) * 100).round(2)

    ozet = pd.DataFrame({
        "Veri Seti": veri_adi,
        "Sutun": eksik.index,
        "Eksik Deger Sayisi": eksik.values,
        "Eksik Deger Orani (%)": oran.values,
    })

    return ozet[ozet["Eksik Deger Sayisi"] > 0]

for name, df in dataframes.items():
    ozet = eksik_deger_ozeti(df, name)
    print(f"\n{name}")
    print("-" * 50)
    if ozet.empty:
        print("Eksik deger bulunmamaktadir.")
    else:
        display(ozet)



Tahsilat
--------------------------------------------------


,Veri Seti,Sutun,Eksik Deger Sayisi,Eksik Deger Orani (%)
5,Tahsilat,Nakit Tahsilat,636470,99.92
6,Tahsilat,Mahsuben Tahsilat,629451,98.82
7,Tahsilat,Kredi Kartı Tahsilatı,636993,100.00
8,Tahsilat,Banka Tahsilatı,8060,1.27



Tahsilat 1
--------------------------------------------------


,Veri Seti,Sutun,Eksik Deger Sayisi,Eksik Deger Orani (%)
6,Tahsilat 1,Son Ödeme Tarihinden Önceki Tahsilat,293724,32.01
7,Tahsilat 1,Son Ödeme Tarihindeki Tahsilat,589439,64.23
8,Tahsilat 1,Son Ödeme (1),896730,97.72
9,Tahsilat 1,Son Ödeme (2),895968,97.64
10,Tahsilat 1,Son Ödeme (3),898739,97.94
11,Tahsilat 1,Son Ödeme (4),900637,98.15
12,Tahsilat 1,Son Ödeme (5),910309,99.20
13,Tahsilat 1,Son Ödeme (6-10),871924,95.02
14,Tahsilat 1,Son Ödeme (10-20),869351,94.74
15,Tahsilat 1,Son Ödeme (20-30),888627,96.84



Hamamozu
--------------------------------------------------
Eksik deger bulunmamaktadir.

Gumushacikoy
--------------------------------------------------
Eksik deger bulunmamaktadir.

Goynucek
--------------------------------------------------
Eksik deger bulunmamaktadir.


## 4. Tahakkuk Verilerinin Birlestirilmesi

Uc ilceye ait tahakkuk verileri tek bir DataFrame altinda birlestirilmis ve birlestirme sonrasi kayit sayisi dogrulanmistir.


In [39]:
df_tahakkuk_all = pd.concat(
    [df_hamamozu, df_gumushacikoy, df_goynucek],
    ignore_index=True
)

toplam_beklenen = len(df_hamamozu) + len(df_gumushacikoy) + len(df_goynucek)

print(f"Hamamozu kayit sayisi     : {len(df_hamamozu):,}")
print(f"Gumushacikoy kayit sayisi : {len(df_gumushacikoy):,}")
print(f"Goynucek kayit sayisi     : {len(df_goynucek):,}")
print("-" * 40)
print(f"Beklenen toplam kayit      : {toplam_beklenen:,}")
print(f"Birlestirilmis kayit sayisi: {len(df_tahakkuk_all):,}")
print("Kayit sayisi dogrulamasi   :", len(df_tahakkuk_all) == toplam_beklenen)


Hamamozu kayit sayisi     : 124,818
Gumushacikoy kayit sayisi : 765,657
Goynucek kayit sayisi     : 295,223
----------------------------------------
Beklenen toplam kayit      : 1,185,698
Birlestirilmis kayit sayisi: 1,185,698
Kayit sayisi dogrulamasi   : True


In [40]:
tarih_sutunlari = [
    "mali_yil_donem",
    "fatura_tarihi",
    "kayit_tarihi",
    "vade_tarihi",
]

for column in tarih_sutunlari:
    df_tahakkuk_all[column] = pd.to_datetime(df_tahakkuk_all[column], errors="coerce")

hesap_sinifi_col = df_tahakkuk_all.columns[8]

df_tahakkuk_all["ilce"] = df_tahakkuk_all["ilce"].str.strip()
df_tahakkuk_all[hesap_sinifi_col] = df_tahakkuk_all[hesap_sinifi_col].str.strip()

print(df_tahakkuk_all[tarih_sutunlari].dtypes)
print("\nTarih sutunlarindaki eksik degerler:")
print(df_tahakkuk_all[tarih_sutunlari].isna().sum())
print("\nIlceler:", df_tahakkuk_all["ilce"].unique())
print("Benzersiz hesap sinifi sayisi:", df_tahakkuk_all[hesap_sinifi_col].nunique())


mali_yil_donem    datetime64[us]
fatura_tarihi     datetime64[us]
kayit_tarihi      datetime64[us]
vade_tarihi       datetime64[us]
dtype: object

Tarih sutunlarindaki eksik degerler:
mali_yil_donem    0
fatura_tarihi     0
kayit_tarihi      0
vade_tarihi       0
dtype: int64

Ilceler: <StringArray>
['HAMAMÖZÜ', 'GÜMÜŞHACIKÖY', 'GÖYNÜCEK']
Length: 3, dtype: str
Benzersiz hesap sinifi sayisi: 37


## 5. Ilce Bazinda Benzersiz Musteri Sayilari

Tahakkuk verilerinde yer alan `sozlesme_hesap_no` degiskeni kullanilarak her ilcedeki benzersiz musteri sayisi hesaplanmistir.


In [41]:
df_musteri_sayilari = (
    df_tahakkuk_all
    .groupby("ilce")["sozlesme_hesap_no"]
    .nunique()
    .sort_values(ascending=False)
    .rename("Benzersiz Musteri Sayisi")
    .reset_index()
    .rename(columns={"ilce": "Ilce"})
)

display(df_musteri_sayilari)


,Ilce,Benzersiz Musteri Sayisi
0,GÜMÜŞHACIKÖY,18190
1,GÖYNÜCEK,7128
2,HAMAMÖZÜ,2981


### Musteri Sayisi Bulgulari

- En yuksek benzersiz musteri sayisi Gumushacikoy ilcesindedir.
- Goynucek ikinci, Hamamozu ucuncu siradadir.
- Ilceler arasinda toplam tuketim karsilastirmasi yapilirken musteri sayisi farki dikkate alinmalidir. Bu nedenle sonraki analizlerde kayit basina veya musteri basina tuketim gostergeleri de onemlidir.


## 6. kWh Tuketim Verisinin Kalite Kontrolu

Birlestirilmis tahakkuk verisindeki `kwh` degiskeni eksik, negatif, sifir ve pozitif degerler acisindan incelenmistir.


In [42]:
toplam_kayit = len(df_tahakkuk_all)
eksik_kwh = df_tahakkuk_all["kwh"].isna().sum()
negatif_kwh = (df_tahakkuk_all["kwh"] < 0).sum()
sifir_kwh = (df_tahakkuk_all["kwh"] == 0).sum()
pozitif_kwh = (df_tahakkuk_all["kwh"] > 0).sum()

print(f"Toplam kayit sayisi : {toplam_kayit:,}")
print(f"Eksik kWh           : {eksik_kwh:,}")
print(f"Negatif kWh         : {negatif_kwh:,}")
print(f"Sifir kWh           : {sifir_kwh:,}")
print(f"Pozitif kWh         : {pozitif_kwh:,}")
print("-" * 40)
print(f"Minimum kWh         : {df_tahakkuk_all['kwh'].min():,.2f}")
print(f"Maksimum kWh        : {df_tahakkuk_all['kwh'].max():,.2f}")


Toplam kayit sayisi : 1,185,698
Eksik kWh           : 0
Negatif kWh         : 151
Sifir kWh           : 55,377
Pozitif kWh         : 1,130,170
----------------------------------------
Minimum kWh         : -25,370.64
Maksimum kWh        : 153,575.73


In [43]:
df_negatif_kwh = df_tahakkuk_all[df_tahakkuk_all["kwh"] < 0].copy()

negatif_ilce = (
    df_negatif_kwh["ilce"]
    .value_counts()
    .rename_axis("Ilce")
    .reset_index(name="Negatif Kayit Sayisi")
)

negatif_hesap_sinifi = (
    df_negatif_kwh[hesap_sinifi_col]
    .value_counts()
    .rename_axis("Hesap Sinifi")
    .reset_index(name="Negatif Kayit Sayisi")
)

display(negatif_ilce)
display(negatif_hesap_sinifi.head(10))


,Ilce,Negatif Kayit Sayisi
0,GÜMÜŞHACIKÖY,107
1,GÖYNÜCEK,40
2,HAMAMÖZÜ,4


,Hesap Sinifi,Negatif Kayit Sayisi
0,Mesken,106
1,Tarımsal Faaliyetler (Kooperatif),16
2,Ticari Faaliyet - Yazıhane,13
3,Tarımsal Faaliyetler (Şahıs),10
4,Şantiye ve Geçici Aboneler,2
5,Resmi Daire,2
6,Belediye,1
7,Süt Toplama Merkezi,1


In [44]:
display(
    df_negatif_kwh[
        ["ilce", "sozlesme_hesap_no", "mali_yil_donem", hesap_sinifi_col, "kwh"]
    ]
    .sort_values("kwh")
    .head(10)
)


,ilce,sozlesme_hesap_no,mali_yil_donem,Hesap Sınıfı,kwh
642557,GÜMÜŞHACIKÖY,3798287663,2025-04-01,Belediye,"-25,370.64"
614593,GÜMÜŞHACIKÖY,5966038883,2023-08-01,Tarımsal Faaliyetler (Kooperatif),"-12,574.78"
408766,GÜMÜŞHACIKÖY,6199137693,2024-08-01,Tarımsal Faaliyetler (Şahıs),"-9,069.13"
398719,GÜMÜŞHACIKÖY,3971569485,2024-08-01,Tarımsal Faaliyetler (Şahıs),"-6,851.44"
1113942,GÖYNÜCEK,7513817453,2025-03-01,Tarımsal Faaliyetler (Kooperatif),"-4,208.64"
866036,GÜMÜŞHACIKÖY,1822905563,2024-10-01,Ticari Faaliyet - Yazıhane,"-3,861.20"
1151085,GÖYNÜCEK,5811626546,2024-10-01,Mesken,"-2,836.64"
866037,GÜMÜŞHACIKÖY,1822905563,2024-10-01,Ticari Faaliyet - Yazıhane,"-2,332.63"
866030,GÜMÜŞHACIKÖY,1822905563,2024-09-01,Ticari Faaliyet - Yazıhane,"-2,315.57"
719646,GÜMÜŞHACIKÖY,9953326179,2024-10-01,Ticari Faaliyet - Yazıhane,"-1,981.66"


### Negatif Tuketim Bulgulari

- Negatif `kwh` kayitlari toplam veri icinde cok dusuk orandadir.
- Negatif degerler otomatik olarak silinmemistir; sayac duzeltmesi, mahsuplasma veya operasyonel bir surecle iliskili olabilir.
- Bu kayitlar sonraki analizlerde ihtiyac halinde ayri filtrelenebilir.


### Negatif ve Sifir Tuketimleri Nasil Yorumluyoruz

Negatif ve sifir `kwh` degerleri dogrudan silinmeden once veri icindeki dagilimlari incelenmistir. Amac, bu kayitlarin rastgele bir veri bozulmasi mi yoksa belirli abone/donem gruplarinda yogunlasan operasyonel kayitlar mi oldugunu anlamaktir.


In [45]:
sifir_kwh_df = df_tahakkuk_all[df_tahakkuk_all["kwh"] == 0].copy()

negatif_sifir_ozet = pd.DataFrame({
    "Gosterge": ["Negatif kWh", "Sifir kWh"],
    "Kayit Sayisi": [len(df_negatif_kwh), len(sifir_kwh_df)],
    "Toplam Veriye Oran (%)": [
        len(df_negatif_kwh) / len(df_tahakkuk_all) * 100,
        len(sifir_kwh_df) / len(df_tahakkuk_all) * 100,
    ],
    "Benzersiz Musteri": [
        df_negatif_kwh["sozlesme_hesap_no"].nunique(),
        sifir_kwh_df["sozlesme_hesap_no"].nunique(),
    ],
}).round(4)

display(negatif_sifir_ozet)

print("Negatif kWh - donem dagilimi")
display(
    df_negatif_kwh["mali_yil_donem"]
    .dt.to_period("M")
    .value_counts()
    .sort_index()
    .rename_axis("Donem")
    .reset_index(name="Kayit Sayisi")
    .tail(12)
)

print("Sifir kWh - ilk 10 hesap sinifi")
display(
    sifir_kwh_df[hesap_sinifi_col]
    .value_counts()
    .rename_axis("Hesap Sinifi")
    .reset_index(name="Sifir Kayit Sayisi")
    .head(10)
)

print("Sifir kWh - ilce dagilimi")
display(
    sifir_kwh_df["ilce"]
    .value_counts()
    .rename_axis("Ilce")
    .reset_index(name="Sifir Kayit Sayisi")
)


,Gosterge,Kayit Sayisi,Toplam Veriye Oran (%),Benzersiz Musteri
0,Negatif kWh,151,0.01,54
1,Sifir kWh,55377,4.67,9883


Negatif kWh - donem dagilimi


,Donem,Kayit Sayisi
18,2024-07,8
19,2024-08,8
20,2024-09,5
21,2024-10,10
22,2024-11,5
23,2024-12,6
24,2025-01,4
25,2025-02,1
26,2025-03,4
27,2025-04,4


Sifir kWh - ilk 10 hesap sinifi


,Hesap Sinifi,Sifir Kayit Sayisi
0,Mesken,47986
1,Ticari Faaliyet - Yazıhane,3102
2,Tarımsal Faaliyetler (Şahıs),1632
3,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,372
4,Şantiye ve Geçici Aboneler,344
5,İbadethane Isıtma/Soğutma/Lojman,323
6,Tarımsal Faaliyetler (Kooperatif),303
7,Köy İçme Suyu Temini ve Dağıtımı Tesisi,223
8,Belediye,150
9,Resmi Daire Lojman,149


Sifir kWh - ilce dagilimi


,Ilce,Sifir Kayit Sayisi
0,GÜMÜŞHACIKÖY,31932
1,GÖYNÜCEK,16824
2,HAMAMÖZÜ,6621


## 7. Aykiri Deger Analizi

`kwh` tuketim degerlerindeki potansiyel aykiri gozlemler ceyrekler acikligi (IQR) yontemi ile incelenmistir.


In [46]:
Q1 = df_tahakkuk_all["kwh"].quantile(0.25)
Q3 = df_tahakkuk_all["kwh"].quantile(0.75)
IQR = Q3 - Q1

alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

df_outliers = df_tahakkuk_all[
    (df_tahakkuk_all["kwh"] < alt_sinir) |
    (df_tahakkuk_all["kwh"] > ust_sinir)
].copy()

print(f"Q1 (25. yuzdelik)      : {Q1:,.2f} kWh")
print(f"Q3 (75. yuzdelik)      : {Q3:,.2f} kWh")
print(f"IQR                    : {IQR:,.2f} kWh")
print(f"Alt sinir              : {alt_sinir:,.2f} kWh")
print(f"Ust sinir              : {ust_sinir:,.2f} kWh")
print("-" * 45)
print(f"Aykiri deger sayisi    : {len(df_outliers):,}")
print(f"Aykiri deger orani     : %{len(df_outliers) / len(df_tahakkuk_all) * 100:.2f}")


Q1 (25. yuzdelik)      : 18.01 kWh
Q3 (75. yuzdelik)      : 80.00 kWh
IQR                    : 61.99 kWh
Alt sinir              : -74.97 kWh
Ust sinir              : 172.98 kWh
---------------------------------------------
Aykiri deger sayisi    : 48,554
Aykiri deger orani     : %4.09


In [47]:
outlier_ilce = (
    df_outliers["ilce"]
    .value_counts()
    .rename_axis("Ilce")
    .reset_index(name="Aykiri Kayit Sayisi")
)

outlier_hesap_sinifi = (
    df_outliers[hesap_sinifi_col]
    .value_counts()
    .rename_axis("Hesap Sinifi")
    .reset_index(name="Aykiri Kayit Sayisi")
)

display(outlier_ilce)
display(outlier_hesap_sinifi.head(10))


,Ilce,Aykiri Kayit Sayisi
0,GÜMÜŞHACIKÖY,31838
1,GÖYNÜCEK,12358
2,HAMAMÖZÜ,4358


,Hesap Sinifi,Aykiri Kayit Sayisi
0,Mesken,20922
1,Ticari Faaliyet - Yazıhane,15818
2,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,2260
3,Tarımsal Faaliyetler (Şahıs),1896
4,Köy İçme Suyu Temini ve Dağıtımı Tesisi,1315
5,Tarımsal Faaliyetler (Kooperatif),997
6,İbadethane Isıtma/Soğutma/Lojman,924
7,Belediye,726
8,Resmi Daire,636
9,Süt Toplama Merkezi,624


In [48]:
display(
    df_tahakkuk_all[
        ["ilce", "sozlesme_hesap_no", "mali_yil_donem", hesap_sinifi_col, "kwh"]
    ]
    .sort_values("kwh", ascending=False)
    .head(10)
)


,ilce,sozlesme_hesap_no,mali_yil_donem,Hesap Sınıfı,kwh
808462,GÜMÜŞHACIKÖY,6414845714,2024-12-01,Lisansız Üreticiler,"153,575.73"
808466,GÜMÜŞHACIKÖY,6414845714,2025-02-01,Lisansız Üreticiler,"150,757.74"
808470,GÜMÜŞHACIKÖY,6414845714,2025-04-01,Lisansız Üreticiler,"136,527.93"
808464,GÜMÜŞHACIKÖY,6414845714,2025-01-01,Lisansız Üreticiler,"134,681.40"
808448,GÜMÜŞHACIKÖY,6414845714,2024-05-01,Lisansız Üreticiler,"124,008.57"
808458,GÜMÜŞHACIKÖY,6414845714,2024-10-01,Lisansız Üreticiler,"119,221.20"
808472,GÜMÜŞHACIKÖY,6414845714,2025-05-01,Lisansız Üreticiler,"116,408.88"
808468,GÜMÜŞHACIKÖY,6414845714,2025-03-01,Lisansız Üreticiler,"111,801.06"
721105,GÜMÜŞHACIKÖY,9150855017,2023-02-01,Sanayi,"109,546.29"
721103,GÜMÜŞHACIKÖY,9150855017,2023-01-01,Sanayi,"109,344.06"


### Aykiri Deger Bulgulari

- IQR yontemi potansiyel aykiri tuketimleri isaretlemek icin kullanilmistir.
- Yuksek tuketimler ozellikle sanayi, lisansiz uretici, belediye ve altyapi odakli hesap siniflarinda dogal olabilir.
- Bu nedenle aykiri degerler otomatik olarak veri setinden cikarilmamis, baglamsal olarak degerlendirilmek uzere korunmustur.


## 8. Hesap Sinifina Gore Tuketim Istatistikleri

Farkli musteri gruplarinin elektrik tuketim profillerini karsilastirmak icin hesap sinifi bazinda temel istatistikler hesaplanmistir.


In [49]:
hesap_sinifi_istatistikleri = (
    df_tahakkuk_all
    .groupby(hesap_sinifi_col)["kwh"]
    .agg(
        Kayit_Sayisi="count",
        Ortalama_kWh="mean",
        Medyan_kWh="median",
        Standart_Sapma_kWh="std"
    )
    .sort_values("Ortalama_kWh", ascending=False)
    .reset_index()
)

hesap_sinifi_istatistikleri[
    ["Ortalama_kWh", "Medyan_kWh", "Standart_Sapma_kWh"]
] = hesap_sinifi_istatistikleri[
    ["Ortalama_kWh", "Medyan_kWh", "Standart_Sapma_kWh"]
].round(2)

display(hesap_sinifi_istatistikleri)


,Hesap Sınıfı,Kayit_Sayisi,Ortalama_kWh,Medyan_kWh,Standart_Sapma_kWh
0,Karayolları Genel Müdürlüğü Aydınlatma,41,"30,203.43","36,810.90","16,963.57"
1,Aritma Tesisleri,35,"16,594.17","16,186.91","11,656.71"
2,Lisansız Üreticiler,180,"16,155.25",322.32,"35,926.42"
3,Sanayi,187,"7,293.80","1,531.68","13,716.34"
4,İçme-Kullanma Suyu (Belediye),417,"5,213.51",400.02,"8,948.07"
5,Tarımsal Faaliyetler (Kooperatif),1632,"3,779.29",282.72,"9,426.94"
6,Lisansız Üreticiler - Resmi Daire,36,"1,195.05",770.62,"1,229.58"
7,"Resmi SAĞLIK KURULUŞLARI,RESMİ SPOR TES.",442,"1,154.74",239.90,"3,804.87"
8,"Resmi Üniversite,Yük.Okul,Kurs,Yurt,Okul",476,883.89,285.41,"1,435.31"
9,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,8500,688.44,23.86,"3,911.91"


### Hesap Sinifi Bulgulari

- Mesken sinifi kayit sayisi bakimindan veri setinin buyuk bolumunu olusturmaktadir.
- Karayollari aydinlatma, aritma tesisleri, lisansiz ureticiler ve sanayi gibi siniflarda kayit basina ortalama tuketim cok daha yuksektir.
- Bazi siniflarda ortalama ve medyan arasindaki fark buyuktur; bu durum dagilimin yuksek tuketimlere dogru carpik oldugunu gosterir.
- Az sayida gozleme sahip hesap siniflarinda ortalama tuketim yorumlanirken orneklem buyuklugu dikkate alinmalidir.


## 9. Analize Hazirlik

Ham veri setleri korunmus, sonraki notebooklarda kullanilmak uzere analiz kopyalari olusturulmustur. Odeme turu ve odeme zamanlamasi sutunlarindaki eksik degerler ilgili kategoride tahsilat bulunmadigi anlamina geldigi icin `0` ile doldurulmustur.


In [50]:
df_tahsilat_clean = df_tahsilat.copy()
df_tahsilat_1_clean = df_tahsilat_1.copy()
df_tahakkuk_clean = df_tahakkuk_all.copy()

odeme_turu_sutunlari = df_tahsilat_clean.columns[5:9].tolist()
odeme_zamani_sutunlari = df_tahsilat_1_clean.columns[6:22].tolist()

df_tahsilat_clean[odeme_turu_sutunlari] = df_tahsilat_clean[odeme_turu_sutunlari].fillna(0)
df_tahsilat_1_clean[odeme_zamani_sutunlari] = df_tahsilat_1_clean[odeme_zamani_sutunlari].fillna(0)


In [51]:
ay_haritasi = {
    "OCK": 1,
    "SBT": 2,
    "SUB": 2,
    "MAR": 3,
    "NIS": 4,
    "MAY": 5,
    "HAZ": 6,
    "TEM": 7,
    "AGU": 8,
    "EYL": 9,
    "EKM": 10,
    "KAS": 11,
    "KSM": 11,
    "ARA": 12,
    "ARL": 12,
}

def turkce_karakterleri_sadelestir(seri):
    return (
        seri.astype("string")
        .str.strip()
        .str.upper()
        .str.replace(chr(350), "S", regex=False)
        .str.replace(chr(286), "G", regex=False)
        .str.replace(chr(220), "U", regex=False)
        .str.replace(chr(214), "O", regex=False)
        .str.replace(chr(304), "I", regex=False)
        .str.replace(chr(199), "C", regex=False)
    )

def turkce_donem_to_datetime(seri):
    parcalar = turkce_karakterleri_sadelestir(seri).str.split(expand=True)
    ay = parcalar[0].map(ay_haritasi)
    yil = pd.to_numeric(parcalar[1], errors="coerce")
    return pd.to_datetime({"year": yil, "month": ay, "day": 1}, errors="coerce")

donem_col = df_tahsilat_1_clean.columns[0]

df_tahsilat_clean["Tahsilat Tarihi"] = pd.to_datetime(
    df_tahsilat_clean["Tahsilat Tarihi"],
    errors="coerce"
)

df_tahsilat_1_clean["donem_tarihi"] = turkce_donem_to_datetime(
    df_tahsilat_1_clean[donem_col]
)

print("Tahsilat tarihi eksik deger:", df_tahsilat_clean["Tahsilat Tarihi"].isna().sum())
print("Donem tarihi eksik deger   :", df_tahsilat_1_clean["donem_tarihi"].isna().sum())


Tahsilat tarihi eksik deger: 0
Donem tarihi eksik deger   : 0


In [52]:
duplicate_count = df_tahakkuk_clean.duplicated().sum()

print(f"Tahakkuk tamamen yinelenen kayit sayisi: {duplicate_count:,}")
print(f"Yinelenen kayit orani: %{duplicate_count / len(df_tahakkuk_clean) * 100:.4f}")
print("\nTemiz kopyalarda kalan toplam eksik degerler:")
print("Tahsilat  :", df_tahsilat_clean.isna().sum().sum())
print("Tahsilat 1:", df_tahsilat_1_clean.isna().sum().sum())
print("Tahakkuk  :", df_tahakkuk_clean.isna().sum().sum())


Tahakkuk tamamen yinelenen kayit sayisi: 0
Yinelenen kayit orani: %0.0000

Temiz kopyalarda kalan toplam eksik degerler:
Tahsilat  : 0
Tahsilat 1: 0
Tahakkuk  : 0


## 10. Genel Degerlendirme

- Uc ilceye ait tahakkuk verileri basariyla birlestirilmis ve toplam kayit sayisi dogrulanmistir.
- Gumushacikoy musteri ve kayit hacmi bakimindan en buyuk ilcedir; bu nedenle toplam tuketim yorumlarinda ilce buyuklugu dikkate alinmalidir.
- `kwh` degiskeninde eksik deger bulunmamaktadir; buna karsin negatif ve sifir tuketim kayitlari tespit edilmistir.
- Negatif tuketimler fiziksel tuketim davranisi olarak yorumlanmamalidir. Ancak veri icindeki sayilari cok dusuk oldugu ve belirli kayitlarda yogunlastigi icin bu degerler dogrudan silinmemis, operasyonel duzeltme/iptal/mahsuplasma ihtimali olan kayitlar olarak isaretlenmistir.
- Sifir tuketimler negatif degerlerden farkli olarak gercek bir musteri durumunu temsil edebilir. Bu nedenle sifir kayitlar da silinmemis, hangi ilce ve hesap siniflarinda yogunlastigi ayrica incelenmistir.
- Bu yaklasimla ham veri korunmus, negatif ve sifir tuketimler ise sonraki analizlerde dahil/haric senaryolar icin ayrilabilecek veri kalitesi isaretleri olarak ele alinmistir.
- IQR yontemiyle potansiyel aykiri degerler belirlenmis ancak hesap siniflari arasindaki dogal tuketim farklari nedeniyle otomatik olarak silinmemistir.
- Tuketim davranisi hesap siniflarina gore belirgin sekilde degismektedir.
- Hazirlanan temiz kopyalar sonraki gorsellestirme ve veri hikayesi notebooklari icin temel olusturacaktir.
